In [ ]:
# Machine Learning on Conductivity Fluctuation Features

This notebook analyzes spectral features extracted from conductivity fluctuation signals.

Material groups used for supervised classification:
- BI — bismuth films
- CR — chromium films
- FE — iron films

Reference group:
- XXX — modal reference film used to identify composition-independent noise

Models:
- K-Means clustering
- K-Nearest Neighbors
- Fully connected neural network

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    silhouette_score,
)

In [ ]:
DATASET_PATH = Path("../data/processed/conductivity_features.csv")
RESULTS_DIR = Path("../results")

REFERENCE_LABEL = "XXX"
MATERIAL_ORDER = ["BI", "CR", "FE"]

RANDOM_STATE = 42

df = pd.read_csv(DATASET_PATH)

df.head()

In [ ]:
df.info()
df["material"].value_counts()

In [ ]:
## Prepare material classification dataset

XXX reference samples are excluded from supervised material classification because they do not represent a target material class.

In [ ]:
df_materials = df[df["material"] != REFERENCE_LABEL].copy()

X = df_materials[["gamma", "energy"]].copy()
y = df_materials["material"].copy()

X["energy"] = np.log10(X["energy"].clip(lower=1e-30))

X.describe()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
n_clusters = y.nunique()

kmeans = KMeans(
    n_clusters=n_clusters,
    random_state=RANDOM_STATE,
    n_init=20,
)

cluster_labels = kmeans.fit_predict(X_train_scaled)

silhouette = silhouette_score(X_train_scaled, cluster_labels)

print(f"Number of clusters: {n_clusters}")
print(f"Silhouette score: {silhouette:.4f}")

In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    x=X_train_scaled[:, 0],
    y=X_train_scaled[:, 1],
    hue=cluster_labels,
    palette="tab10",
    s=25,
)

plt.xlabel("Gamma (scaled)")
plt.ylabel("Log10 energy (scaled)")
plt.title("K-Means clustering of conductivity fluctuation features")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "kmeans_clusters.png", dpi=300)
plt.show()

In [ ]:
knn = KNeighborsClassifier(
    n_neighbors=5,
    weights="distance",
)

knn.fit(X_train_scaled, y_train)

knn_pred = knn.predict(X_test_scaled)

knn_accuracy = accuracy_score(y_test, knn_pred)

print(f"KNN accuracy: {knn_accuracy * 100:.2f}%")
print(
    classification_report(
        y_test,
        knn_pred,
        labels=MATERIAL_ORDER,
        target_names=MATERIAL_ORDER,
        zero_division=0,
    )
)

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    max_iter=1000,
    early_stopping=True,
    random_state=RANDOM_STATE,
)

mlp.fit(X_train_scaled, y_train)

mlp_pred = mlp.predict(X_test_scaled)

mlp_accuracy = accuracy_score(y_test, mlp_pred)

print(f"Neural network accuracy: {mlp_accuracy * 100:.2f}%")
print(
    classification_report(
        y_test,
        mlp_pred,
        labels=MATERIAL_ORDER,
        target_names=MATERIAL_ORDER,
        zero_division=0,
    )
)

In [ ]:
cm = confusion_matrix(y_test, mlp_pred, labels=MATERIAL_ORDER)

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=MATERIAL_ORDER,
    yticklabels=MATERIAL_ORDER,
)

plt.xlabel("Predicted material")
plt.ylabel("True material")
plt.title("Neural network confusion matrix")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "neural_network_confusion_matrix.png", dpi=300)
plt.show()

In [ ]:
## Reference film analysis

XXX is a modal reference film. It is used to identify conductivity fluctuation components that are not related to the chemical composition of the material films.

In [ ]:
df_reference = df[df["material"] == REFERENCE_LABEL].copy()

print(f"Reference film samples: {len(df_reference)}")
print(f"Material film samples: {len(df_materials)}")

plt.figure(figsize=(10, 6))

sns.kdeplot(
    data=df_materials,
    x="gamma",
    hue="material",
    common_norm=False,
    fill=True,
    alpha=0.25,
)

sns.kdeplot(
    data=df_reference,
    x="gamma",
    color="black",
    linewidth=2,
    label="XXX reference film",
)

plt.xlabel("Gamma")
plt.ylabel("Density")
plt.title("Gamma distribution: material films vs reference film")
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "reference_vs_materials_gamma.png", dpi=300)
plt.show()

In [ ]:
df_reference_plot = df_reference.copy()
df_materials_plot = df_materials.copy()

df_reference_plot["log_energy"] = np.log10(
    df_reference_plot["energy"].clip(lower=1e-30)
)

df_materials_plot["log_energy"] = np.log10(
    df_materials_plot["energy"].clip(lower=1e-30)
)

plt.figure(figsize=(10, 6))

sns.kdeplot(
    data=df_materials_plot,
    x="log_energy",
    hue="material",
    common_norm=False,
    fill=True,
    alpha=0.25,
)

sns.kdeplot(
    data=df_reference_plot,
    x="log_energy",
    color="black",
    linewidth=2,
    label="XXX reference film",
)

plt.xlabel("Log10 spectral energy")
plt.ylabel("Density")
plt.title("Energy distribution: material films vs reference film")
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "reference_vs_materials_energy.png", dpi=300)
plt.show()

In [ ]:
metrics = pd.DataFrame(
    [
        {
            "model": "K-Means",
            "metric": "silhouette_score",
            "value": silhouette,
        },
        {
            "model": "KNN",
            "metric": "accuracy",
            "value": knn_accuracy,
        },
        {
            "model": "Neural network",
            "metric": "accuracy",
            "value": mlp_accuracy,
        },
    ]
)

metrics.to_csv(RESULTS_DIR / "model_metrics.csv", index=False)

metrics

In [ ]:
## Results

- XXX reference samples were used to analyze composition-independent noise.
- BI, CR, and FE samples were used for supervised material classification.
- K-Means was used for exploratory clustering.
- KNN and a fully connected neural network were used for material classification.
- Model performance was evaluated using accuracy, classification reports, and confusion matrices.

The thesis pipeline achieved 89.41% recognition accuracy.  
Results in this reconstructed notebook may differ because the original final code and full experimental dataset were not preserved.